# Train & Export SVG `feConvolveMatrix` MNIST CNN (`nojs/cnn`)

This notebook trains a **BatchNorm-free Convolutional Neural Network** on MNIST designed specifically for **zero-JavaScript HTML5 SVG `<filter>` inference** (`<feConvolveMatrix>`, `<feColorMatrix>`, `<feComposite>`, `<feMorphology>`, and `<feComponentTransfer>`), then exports the learned weights into `weights.json`.

### Why Quantization-Aware SVG Architecture?
1. **No BatchNorm**: Keeps every spatial convolution a pure `weight * x + bias` operation directly expressible in `<feConvolveMatrix>`.
2. **Native `[0, 1]` Clamped Activations & 8-Bit Buffers**: Browser SVG filter surfaces (`color-interpolation-filters="sRGB"`) clamp intermediate outputs to `[0.0, 1.0]` at 8-bit (`uint8`, `1/255`) precision. Using `clamp(x, 0, 1)` with a Straight-Through Estimator (`round(x * 255) / 255`) turns SVG clamping into the network's native nonlinearity and eliminates float-to-SVG quantization drop.
3. **Signed Multi-Channel Mixing without Clipping**: Cross-channel `1x1` projections (`<feColorMatrix>`) encode signed partial sums around a `+0.5` zero-point (`<feComposite operator="arithmetic" k2="1" k3="1" k4="-0.5"/>`) with guaranteed $L_1$-bounded scaling so negative weights never clip.
4. **Zero-Boundary Spatial Coordinate Chain (`order <= 9x9`)**:
   - Input `28x28` (`0..27`) $\rightarrow$ `Conv1` (`5x5`, valid `{2..25}`) $\rightarrow$ `Pool1` (`2x2`, stride-2 `{2,4,...,24}`)
   - $\rightarrow$ `Mix2` (`12->15`) + `DWConv2` (`3x3` dilated-2 = `order 5x5`, `{4,6,...,22}`) $\rightarrow$ `Pool2` (`2x2` dilated-2 = `order 3x3`, `{4,8,12,16,20}`)
   - $\rightarrow$ `Mix3` (`15->24`) + `DWConv3` (`3x3` dilated-4 = `order 9x9`, `{8,12,16}`)
   - $\rightarrow$ `DWConv4` (`3x3` dilated-4 = `order 9x9` at center pixel `(12, 12)`) $\rightarrow$ `FC (24->10)` Logits $\rightarrow$ Exact 8-bit `LogSumExp` **Softmax**.

**Usage in Google Colab**: Click **Runtime $\rightarrow$ Run all**. Training takes ~15 seconds on a T4 GPU (~45 seconds on CPU) and automatically downloads `weights.json` when finished.

In [ ]:
import gzip
import json
import math
import os
import random
import struct
import urllib.request
from copy import deepcopy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | Device: {device}")

In [ ]:
# ---------------------------------------------------------------------------
# 1. Download MNIST & Create Stratified Train / Validation / Test Splits
# ---------------------------------------------------------------------------

def load_mnist_tensors(data_dir: str = "./data"):
    """Download MNIST via torchvision with a direct GCS mirror fallback."""
    os.makedirs(data_dir, exist_ok=True)
    try:
        import torchvision
        train_ds = torchvision.datasets.MNIST(root=data_dir, train=True, download=True)
        test_ds = torchvision.datasets.MNIST(root=data_dir, train=False, download=True)
        x_train_all = train_ds.data.numpy().astype(np.float32) / 255.0
        y_train_all = train_ds.targets.numpy().astype(np.int64)
        x_test = test_ds.data.numpy().astype(np.float32) / 255.0
        y_test = test_ds.targets.numpy().astype(np.int64)
        return x_train_all, y_train_all, x_test, y_test
    except Exception as exc:
        print(f"torchvision download fallback triggered ({exc}); fetching from GCS mirror...")
        base_url = "https://storage.googleapis.com/cvdf-datasets/mnist/"
        files = {
            "train_img": "train-images-idx3-ubyte.gz",
            "train_lbl": "train-labels-idx1-ubyte.gz",
            "test_img": "t10k-images-idx3-ubyte.gz",
            "test_lbl": "t10k-labels-idx1-ubyte.gz",
        }
        paths = {}
        for k, fname in files.items():
            p = os.path.join(data_dir, fname)
            if not os.path.exists(p):
                urllib.request.urlretrieve(base_url + fname, p)
            paths[k] = p

        def read_idx_images(path):
            with gzip.open(path, "rb") as f:
                _, n, rows, cols = struct.unpack(">IIII", f.read(16))
                return np.frombuffer(f.read(), dtype=np.uint8).reshape(n, rows, cols).astype(np.float32) / 255.0

        def read_idx_labels(path):
            with gzip.open(path, "rb") as f:
                _, n = struct.unpack(">II", f.read(8))
                return np.frombuffer(f.read(), dtype=np.uint8).astype(np.int64)

        return (
            read_idx_images(paths["train_img"]),
            read_idx_labels(paths["train_lbl"]),
            read_idx_images(paths["test_img"]),
            read_idx_labels(paths["test_lbl"]),
        )


x_all, y_all, x_test_np, y_test_np = load_mnist_tensors()

# Stratified 54,000 train / 6,000 validation split (600 per digit class in val)
rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for digit in range(10):
    cls_idx = np.where(y_all == digit)[0]
    rng.shuffle(cls_idx)
    val_indices.append(cls_idx[:600])
    train_indices.append(cls_idx[600:])

train_idx = np.concatenate(train_indices)
val_idx = np.concatenate(val_indices)
rng.shuffle(train_idx)
rng.shuffle(val_idx)

x_train_np, y_train_np = x_all[train_idx], y_all[train_idx]
x_val_np, y_val_np = x_all[val_idx], y_all[val_idx]

print(f"Train split:      {x_train_np.shape[0]:,} images ({x_train_np.shape[1]}x{x_train_np.shape[2]})")
print(f"Validation split: {x_val_np.shape[0]:,} images (600 per class)")
print(f"Test set:         {x_test_np.shape[0]:,} images")

BATCH_SIZE = 256
train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_train_np[:, None, :, :]), torch.from_numpy(y_train_np)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=False,
)
val_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_val_np[:, None, :, :]), torch.from_numpy(y_val_np)),
    batch_size=512,
    shuffle=False,
)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(x_test_np[:, None, :, :]), torch.from_numpy(y_test_np)),
    batch_size=512,
    shuffle=False,
)


def augment_ocr_batch(x: torch.Tensor) -> torch.Tensor:
    """Fast GPU/CPU batch augmentation for real-world OCR generalization.
    Applies subtle rotation/scale/shift jitter and paper-contrast variations
    to 75% of training samples while leaving 25% crisp MNIST.
    """
    b = x.shape[0]
    dev = x.device
    apply_mask = (torch.rand(b, 1, 1, 1, device=dev) < 0.75)

    angles = (torch.rand(b, device=dev) - 0.5) * (2.0 * math.radians(9.0))
    scales = 0.86 + torch.rand(b, device=dev) * 0.26  # [0.86, 1.12] inverse zoom
    tx = (torch.rand(b, device=dev) - 0.5) * 0.16     # ~+-2.2 px
    ty = (torch.rand(b, device=dev) - 0.5) * 0.16

    cos_a = torch.cos(angles) / scales
    sin_a = torch.sin(angles) / scales
    theta = torch.zeros(b, 2, 3, device=dev, dtype=x.dtype)
    theta[:, 0, 0] = cos_a
    theta[:, 0, 1] = -sin_a
    theta[:, 0, 2] = tx
    theta[:, 1, 0] = sin_a
    theta[:, 1, 1] = cos_a
    theta[:, 1, 2] = ty

    grid = F.affine_grid(theta, size=x.shape, align_corners=False)
    warped = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)

    # Mild contrast & off-black background floor jitter (models real paper scans)
    gain = 0.72 + 0.36 * torch.rand(b, 1, 1, 1, device=dev)
    bg_lift = 0.10 * torch.rand(b, 1, 1, 1, device=dev) * (torch.rand(b, 1, 1, 1, device=dev) < 0.35)
    warped = torch.clamp(warped * gain + bg_lift, 0.0, 1.0)

    return torch.where(apply_mask, warped, x)


# Preview raw vs OCR-augmented samples
fig, axes = plt.subplots(2, 8, figsize=(11, 3.0))
sample_raw = torch.from_numpy(x_train_np[:8, None, :, :])
sample_aug = augment_ocr_batch(sample_raw.clone())
for i in range(8):
    axes[0, i].imshow(sample_raw[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"Raw: {y_train_np[i]}", fontsize=9)
    axes[0, i].axis("off")
    axes[1, i].imshow(sample_aug[i, 0], cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title("OCR Aug", fontsize=9)
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 2. BatchNorm-Free SVG-Exact CNN Architecture (8-Bit QAT)
# ---------------------------------------------------------------------------

def ste_q8(x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
    """SVG intermediate buffer physics: clamp to [0, 1] + 8-bit uint8 STE rounding."""
    xc = torch.clamp(x, 0.0, 1.0)
    if not quantize:
        return xc
    xq = torch.round(xc * 255.0) / 255.0
    return xc + (xq - xc).detach()


class SVGPointwise1x1(nn.Module):
    """Pointwise 1x1 cross-channel mixing matching SVG <feColorMatrix> RGB triplets
    summed around a +0.5 zero-point via <feComposite k2=1 k3=1 k4=-0.5>.
    """

    def __init__(self, in_channels: int, out_channels: int, init_gain: float = 1.0):
        super().__init__()
        assert in_channels % 3 == 0 and out_channels % 3 == 0
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels))
        self.bias = nn.Parameter(torch.zeros(out_channels))
        nn.init.kaiming_normal_(self.weight, mode="fan_in", nonlinearity="relu")
        with torch.no_grad():
            self.weight.mul_(init_gain)
            self.bias.fill_(0.02)

    def compute_triplet_scales(self) -> torch.Tensor:
        """Per-output-RGB-triplet L1 bound ensuring 0.5-biased sums stay inside [0.045, 0.955]."""
        with torch.no_grad():
            w_grouped = self.weight.view(self.out_channels // 3, 3, self.in_channels)
            l1_max = w_grouped.abs().sum(dim=2).amax(dim=1)  # (G_out,)
            return torch.clamp(2.2 * l1_max, min=1.0)

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        g_in = self.in_channels // 3
        g_out = self.out_channels // 3
        scales = self.compute_triplet_scales()  # (g_out,)
        s_full = scales.repeat_interleave(3).view(1, self.out_channels, 1, 1)

        # Group 0: <feColorMatrix> with +0.5 zero-point
        w0 = (self.weight[:, 0:3] / scales.repeat_interleave(3).unsqueeze(1)).view(self.out_channels, 3, 1, 1)
        acc = ste_q8(0.5 + F.conv2d(x[:, 0:3], w0), quantize=quantize)

        # Remaining RGB groups: <feColorMatrix> + <feComposite k2=1 k3=1 k4=-0.5>
        for g in range(1, g_in):
            wg = (
                self.weight[:, 3 * g : 3 * (g + 1)] / scales.repeat_interleave(3).unsqueeze(1)
            ).view(self.out_channels, 3, 1, 1)
            part = ste_q8(0.5 + F.conv2d(x[:, 3 * g : 3 * (g + 1)], wg), quantize=quantize)
            acc = ste_q8(acc + part - 0.5, quantize=quantize)

        # <feComponentTransfer type="linear" slope="S" intercept="b - 0.5*S">
        b_view = self.bias.view(1, self.out_channels, 1, 1)
        return ste_q8(s_full * acc + (b_view - 0.5 * s_full), quantize=quantize)


class SVGConvNet(nn.Module):
    """BatchNorm-free All-Conv + Pointwise CNN tailored for HTML5 SVG <feConvolveMatrix>."""

    def __init__(self):
        super().__init__()
        # Stage 1: 1 -> 12, 5x5 valid conv (28x28 -> 24x24) + 2x2 avg pool (-> 12x12)
        self.conv1 = nn.Conv2d(1, 12, kernel_size=5, stride=1, padding=0, bias=True)

        # Stage 2: 12 -> 15 pointwise + 3x3 depthwise valid (12x12 -> 10x10) + 2x2 avg pool (-> 5x5)
        self.mix2 = SVGPointwise1x1(12, 15, init_gain=1.25)
        self.dwconv2 = nn.Conv2d(15, 15, kernel_size=3, stride=1, padding=0, groups=15, bias=True)

        # Stage 3: 15 -> 24 pointwise + 3x3 depthwise valid (5x5 -> 3x3)
        self.mix3 = SVGPointwise1x1(15, 24, init_gain=1.25)
        self.dwconv3 = nn.Conv2d(24, 24, kernel_size=3, stride=1, padding=0, groups=24, bias=True)

        # Stage 4: 24 -> 24 3x3 depthwise global spatial aggregator (3x3 -> 1x1 at pixel (12, 12))
        self.dwconv4 = nn.Conv2d(24, 24, kernel_size=3, stride=1, padding=0, groups=24, bias=True)

        # Stage 5: Linear(24 -> 10) logit head
        self.fc = nn.Linear(24, 10, bias=True)

        for m in (self.conv1, self.dwconv2, self.dwconv3, self.dwconv4):
            nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
            with torch.no_grad():
                m.bias.fill_(0.02)

    def compute_logit_half_range(self) -> torch.Tensor:
        with torch.no_grad():
            max_bound = (self.fc.bias.abs() + self.fc.weight.abs().sum(dim=1)).amax()
            return torch.clamp(1.15 * max_bound, min=8.0)

    def forward(self, x: torch.Tensor, quantize: bool = True) -> torch.Tensor:
        x = ste_q8(x, quantize=quantize)                                    # (B, 1, 28, 28)
        x = ste_q8(self.conv1(x), quantize=quantize)                        # (B, 12, 24, 24)
        x = ste_q8(F.avg_pool2d(x, 2, 2), quantize=quantize)                # (B, 12, 12, 12)

        x = self.mix2(x, quantize=quantize)                                 # (B, 15, 12, 12)
        x = ste_q8(self.dwconv2(x), quantize=quantize)                      # (B, 15, 10, 10)
        x = ste_q8(F.avg_pool2d(x, 2, 2), quantize=quantize)                # (B, 15, 5, 5)

        x = self.mix3(x, quantize=quantize)                                 # (B, 24, 5, 5)
        x = ste_q8(self.dwconv3(x), quantize=quantize)                      # (B, 24, 3, 3)
        x = ste_q8(self.dwconv4(x), quantize=quantize)                      # (B, 24, 1, 1)

        h = x.flatten(1)                                                    # (B, 24)
        if not quantize:
            return self.fc(h)

        # Simulate 8-group RGB <feColorMatrix> encoded logit accumulation u = 0.5 + z / (2M)
        m_half = self.compute_logit_half_range()
        two_m = 2.0 * m_half
        acc = ste_q8(0.5 + (self.fc.bias.unsqueeze(0) + h[:, 0:3] @ self.fc.weight[:, 0:3].t()) / two_m, quantize=True)
        for g in range(1, 8):
            part = ste_q8(0.5 + (h[:, 3 * g : 3 * (g + 1)] @ self.fc.weight[:, 3 * g : 3 * (g + 1)].t()) / two_m, quantize=True)
            acc = ste_q8(acc + part - 0.5, quantize=True)
        return (acc - 0.5) * two_m


model = SVGConvNet().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"SVGConvNet initialized | Total trainable parameters: {total_params:,} (0 BatchNorm layers)")

In [ ]:
# ---------------------------------------------------------------------------
# 3. Train with Backprop & Select Best Checkpoint on Validation Split
# ---------------------------------------------------------------------------

EPOCHS = 14
optimizer = torch.optim.AdamW(model.parameters(), lr=3.5e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-4)

@torch.no_grad()
def evaluate(loader: DataLoader, quantize: bool = True):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb, quantize=quantize)
        loss = F.cross_entropy(logits, yb)
        total_loss += loss.item() * yb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)
    return total_loss / total, correct / total


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    # Warm up first 2 epochs in smooth [0,1]-clamped float, then enable full 8-bit STE QAT
    use_qat = (epoch >= 3)
    run_loss, run_correct, run_total = 0.0, 0, 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        xb_aug = augment_ocr_batch(xb)

        optimizer.zero_grad()
        logits = model(xb_aug, quantize=use_qat)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()

        run_loss += loss.item() * yb.size(0)
        run_correct += (logits.argmax(dim=1) == yb).sum().item()
        run_total += yb.size(0)

    scheduler.step()
    train_loss = run_loss / run_total
    train_acc = run_correct / run_total
    val_loss, val_acc = evaluate(val_loader, quantize=True)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    marker = ""
    if val_acc >= best_val_acc:
        best_val_acc = val_acc
        best_state = deepcopy(model.state_dict())
        marker = " * [best]"

    mode_lbl = "8-bit QAT" if use_qat else "Warmup   "
    print(
        f"Epoch {epoch:02d}/{EPOCHS:02d} ({mode_lbl}) | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc * 100:5.2f}% | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc * 100:5.2f}%{marker}"
    )

model.load_state_dict(best_state)
print(f"\nLoaded best validation checkpoint: {best_val_acc * 100:.2f}% validation accuracy")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
epochs_x = range(1, EPOCHS + 1)
ax1.plot(epochs_x, history["train_loss"], label="Train Loss")
ax1.plot(epochs_x, history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-Entropy Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_x, [a * 100 for a in history["train_acc"]], label="Train Acc (%)")
ax2.plot(epochs_x, [a * 100 for a in history["val_acc"]], label="Val QAT Acc (%)")
ax2.axhline(90.0, color="crimson", linestyle="--", alpha=0.6, label="90% Target")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Training & Validation Accuracy")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 4. Translate Trained Weights into SVG <feConvolveMatrix> & <feColorMatrix>
#    + Bit-Exact 8-Bit SVG Filter Simulator in Pure NumPy
# ---------------------------------------------------------------------------

def dilate_and_flip_kernel(k2d: np.ndarray, dilation: int) -> list[float]:
    """Embed a (H, W) cross-correlation kernel into a dilated grid and flip 180 deg
    so W3C <feConvolveMatrix> evaluates exact cross-correlation.
    """
    kh, kw = k2d.shape
    oh = (kh - 1) * dilation + 1
    ow = (kw - 1) * dilation + 1
    expanded = np.zeros((oh, ow), dtype=np.float64)
    expanded[::dilation, ::dilation] = k2d
    flipped = expanded[::-1, ::-1]
    return [round(float(v), 6) for v in flipped.reshape(-1)]


def export_conv_stage(conv: nn.Conv2d, dilation: int) -> dict:
    w = conv.weight.detach().cpu().numpy()  # (C_out, 1, kH, kW)
    b = conv.bias.detach().cpu().numpy()    # (C_out,)
    kh, kw = w.shape[2], w.shape[3]
    oh = (kh - 1) * dilation + 1
    ow = (kw - 1) * dilation + 1
    tx = (ow - 1) // 2
    ty = (oh - 1) // 2
    filters = []
    for c in range(w.shape[0]):
        filters.append({
            "channel": c,
            "order": [ow, oh],
            "targetX": tx,
            "targetY": ty,
            "divisor": 1.0,
            "bias": round(float(b[c]), 6),
            "kernelMatrix": dilate_and_flip_kernel(w[c, 0], dilation=dilation),
            "weights_unflipped_3x3_or_5x5": [[round(float(x), 6) for x in row] for row in w[c, 0]],
        })
    return {
        "num_channels": int(w.shape[0]),
        "base_kernel_size": [kh, kw],
        "dilation": dilation,
        "svg_order": [ow, oh],
        "targetX": tx,
        "targetY": ty,
        "filters": filters,
    }


def export_pointwise_stage(pw: SVGPointwise1x1) -> dict:
    w = pw.weight.detach().cpu().numpy()
    b = pw.bias.detach().cpu().numpy()
    scales = pw.compute_triplet_scales().cpu().numpy()
    g_in = pw.in_channels // 3
    g_out = pw.out_channels // 3

    triplets = []
    for o in range(g_out):
        s_o = float(scales[o])
        b_slice = b[3 * o : 3 * (o + 1)]
        color_matrices = []
        for g in range(g_in):
            blk = w[3 * o : 3 * (o + 1), 3 * g : 3 * (g + 1)] / s_o
            # 4x5 SVG feColorMatrix values (rows R, G, B, A; cols R, G, B, A, offset)
            mat20 = [
                round(float(blk[0, 0]), 6), round(float(blk[0, 1]), 6), round(float(blk[0, 2]), 6), 0.0, 0.5,
                round(float(blk[1, 0]), 6), round(float(blk[1, 1]), 6), round(float(blk[1, 2]), 6), 0.0, 0.5,
                round(float(blk[2, 0]), 6), round(float(blk[2, 1]), 6), round(float(blk[2, 2]), 6), 0.0, 0.5,
                0.0, 0.0, 0.0, 0.0, 1.0,
            ]
            color_matrices.append(mat20)
        triplets.append({
            "out_triplet": o,
            "scale": round(s_o, 6),
            "rescale_slopes": [round(s_o, 6)] * 3,
            "rescale_intercepts": [round(float(b_slice[r] - 0.5 * s_o), 6) for r in range(3)],
            "color_matrices": color_matrices,
        })
    return {
        "in_channels": pw.in_channels,
        "out_channels": pw.out_channels,
        "triplets": triplets,
    }


def export_fc_and_softmax(model: SVGConvNet, softmax_window: float = 7.0, lut_size: int = 65) -> dict:
    w_fc = model.fc.weight.detach().cpu().numpy()  # (10, 24)
    b_fc = model.fc.bias.detach().cpu().numpy()    # (10,)
    m_half = float(model.compute_logit_half_range().cpu().item())
    two_m = 2.0 * m_half

    # Pad 10 classes to 12 (4 RGB triplets: digits 0..2, 3..5, 6..8, 9+unused)
    w_pad = np.zeros((12, 24), dtype=np.float64)
    b_pad = np.zeros(12, dtype=np.float64)
    w_pad[:10] = w_fc
    b_pad[:10] = b_fc

    triplets = []
    for o in range(4):
        color_matrices = []
        for g in range(8):
            blk = w_pad[3 * o : 3 * (o + 1), 3 * g : 3 * (g + 1)] / two_m
            off = [0.5 + (float(b_pad[3 * o + r]) / two_m if g == 0 else 0.0) for r in range(3)]
            mat20 = [
                round(float(blk[0, 0]), 6), round(float(blk[0, 1]), 6), round(float(blk[0, 2]), 6), 0.0, round(off[0], 6),
                round(float(blk[1, 0]), 6), round(float(blk[1, 1]), 6), round(float(blk[1, 2]), 6), 0.0, round(off[1], 6),
                round(float(blk[2, 0]), 6), round(float(blk[2, 1]), 6), round(float(blk[2, 2]), 6), 0.0, round(off[2], 6),
                0.0, 0.0, 0.0, 0.0, 1.0,
            ]
            color_matrices.append(mat20)
        triplets.append({"out_triplet": o, "color_matrices": color_matrices})

    grid = np.linspace(0.0, 1.0, lut_size)
    exp_lut = np.exp(softmax_window * (grid - 1.0))
    exp_lut[0] = 0.0
    logsum_lut = np.log(np.maximum(10.0 * grid, 1.0)) / math.log(10.0)

    return {
        "fc": {
            "in_features": 24,
            "num_classes": 10,
            "logit_half_range_M": round(m_half, 6),
            "anchor_xy": [12, 12],
            "triplets": triplets,
        },
        "softmax": {
            "window_W": softmax_window,
            "diff_scale_r": round(two_m / softmax_window, 6),
            "logsum_sub_k": round(math.log(10.0) / softmax_window, 6),
            "exp_table_values": [round(float(v), 6) for v in exp_lut],
            "logsum_table_values": [round(float(v), 6) for v in logsum_lut],
        },
    }


def build_export_dict(model: SVGConvNet) -> dict:
    fc_sm = export_fc_and_softmax(model)
    return {
        "format": "nojs-svg-feconvolvematrix-cnn-v1",
        "input_size": [28, 28],
        "coordinate_schedule": {
            "input": "0..27 (28x28)",
            "conv1": "2..25 (24x24, stride 1)",
            "pool1": "2,4,..,24 (12x12, stride 2)",
            "dwconv2": "4,6,..,22 (10x10, stride 2)",
            "pool2": "4,8,12,16,20 (5x5, stride 4)",
            "dwconv3": "8,12,16 (3x3, stride 4)",
            "dwconv4": "(12, 12) (1x1 global feature pixel)",
        },
        "layers": {
            "conv1": export_conv_stage(model.conv1, dilation=1),
            "pool1": {"order": [2, 2], "targetX": 0, "targetY": 0, "divisor": 4.0, "kernelMatrix": [1, 1, 1, 1]},
            "mix2": export_pointwise_stage(model.mix2),
            "dwconv2": export_conv_stage(model.dwconv2, dilation=2),
            "pool2": {"order": [3, 3], "targetX": 0, "targetY": 0, "divisor": 4.0, "kernelMatrix": [1, 0, 1, 0, 0, 0, 1, 0, 1]},
            "mix3": export_pointwise_stage(model.mix3),
            "dwconv3": export_conv_stage(model.dwconv3, dilation=4),
            "dwconv4": export_conv_stage(model.dwconv4, dilation=4),
            "fc": fc_sm["fc"],
            "softmax": fc_sm["softmax"],
        },
    }


# --- Bit-Exact Vectorized 8-Bit SVG <filter> Simulator ---

def q8_np(x: np.ndarray) -> np.ndarray:
    return np.round(np.clip(x, 0.0, 1.0) * 255.0) / 255.0


def svg_fe_convolve_batch(buf: np.ndarray, order: list[int], tx: int, ty: int,
                          kernel_flat: list[float], divisor: float, bias: float) -> np.ndarray:
    """Exact W3C <feConvolveMatrix edgeMode='none' preserveAlpha='true'> on (B, 28, 28)."""
    ow, oh = order
    k = np.array(kernel_flat, dtype=np.float64).reshape(oh, ow)
    bsz, h, w = buf.shape
    padded = np.pad(buf, ((0, 0), (ty, oh - 1 - ty), (tx, ow - 1 - tx)), mode="constant", constant_values=0.0)
    acc = np.zeros((bsz, h, w), dtype=np.float64)
    # W3C formula flips kernel indices: K[oh - 1 - i, ow - 1 - j]
    for i in range(oh):
        for j in range(ow):
            kv = k[oh - 1 - i, ow - 1 - j]
            if kv != 0.0:
                acc += kv * padded[:, i : i + h, j : j + w]
    return q8_np(acc / divisor + bias)


def svg_pointwise_batch(ch_bufs: np.ndarray, pw_spec: dict) -> np.ndarray:
    """Exact <feColorMatrix> + <feComposite k2=1 k3=1 k4=-0.5> + <feComponentTransfer> on (B, C_in, 28, 28)."""
    bsz, _, h, w = ch_bufs.shape
    out = np.zeros((bsz, pw_spec["out_channels"], h, w), dtype=np.float64)
    for trip in pw_spec["triplets"]:
        o = trip["out_triplet"]
        acc = None
        for g, mat20 in enumerate(trip["color_matrices"]):
            m = np.array(mat20, dtype=np.float64).reshape(4, 5)[:3, :]
            rgb_in = ch_bufs[:, 3 * g : 3 * (g + 1)]  # (B, 3, H, W)
            part = q8_np(np.einsum("rc,bchw->brhw", m[:, :3], rgb_in) + m[:, 4].reshape(1, 3, 1, 1))
            acc = part if acc is None else q8_np(acc + part - 0.5)
        slopes = np.array(trip["rescale_slopes"], dtype=np.float64).reshape(1, 3, 1, 1)
        intercepts = np.array(trip["rescale_intercepts"], dtype=np.float64).reshape(1, 3, 1, 1)
        out[:, 3 * o : 3 * (o + 1)] = q8_np(acc * slopes + intercepts)
    return out


def svg_lut_eval(x: np.ndarray, table_vals: list[float]) -> np.ndarray:
    tbl = np.array(table_vals, dtype=np.float64)
    n = len(tbl) - 1
    pos = np.clip(x, 0.0, 1.0) * n
    idx = np.minimum(np.floor(pos).astype(np.int64), n - 1)
    frac = pos - idx
    return q8_np(tbl[idx] * (1.0 - frac) + tbl[idx + 1] * frac)


def simulate_svg_cnn_batch(images_float: np.ndarray, spec: dict) -> tuple[np.ndarray, np.ndarray]:
    """Run exact 8-bit SVG filter inference on (N, 28, 28) images in [0, 1].
    Returns (decoded_logits (N, 10), svg_softmax_probs (N, 10)).
    """
    layers = spec["layers"]
    x0 = q8_np(images_float)  # (B, 28, 28)

    # Stage 1: Conv1 + Pool1
    c1 = np.stack([
        svg_fe_convolve_batch(x0, f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in layers["conv1"]["filters"]
    ], axis=1)
    p1_spec = layers["pool1"]
    p1 = np.stack([
        svg_fe_convolve_batch(c1[:, c], p1_spec["order"], p1_spec["targetX"], p1_spec["targetY"], p1_spec["kernelMatrix"], p1_spec["divisor"], 0.0)
        for c in range(c1.shape[1])
    ], axis=1)

    # Stage 2: Mix2 + DWConv2 + Pool2
    m2 = svg_pointwise_batch(p1, layers["mix2"])
    c2 = np.stack([
        svg_fe_convolve_batch(m2[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in layers["dwconv2"]["filters"]
    ], axis=1)
    p2_spec = layers["pool2"]
    p2 = np.stack([
        svg_fe_convolve_batch(c2[:, c], p2_spec["order"], p2_spec["targetX"], p2_spec["targetY"], p2_spec["kernelMatrix"], p2_spec["divisor"], 0.0)
        for c in range(c2.shape[1])
    ], axis=1)

    # Stage 3 & 4: Mix3 + DWConv3 + DWConv4 -> anchor pixel (12, 12)
    m3 = svg_pointwise_batch(p2, layers["mix3"])
    c3 = np.stack([
        svg_fe_convolve_batch(m3[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in layers["dwconv3"]["filters"]
    ], axis=1)
    c4 = np.stack([
        svg_fe_convolve_batch(c3[:, f["channel"]], f["order"], f["targetX"], f["targetY"], f["kernelMatrix"], f["divisor"], f["bias"])
        for f in layers["dwconv4"]["filters"]
    ], axis=1)

    ax, ay = layers["fc"]["anchor_xy"]
    h_anchor = c4[:, :, ay, ax]  # (B, 24)

    # Stage 5: FC 1x1 ColorMatrix Readout at (12, 12)
    u_pad = np.zeros((x0.shape[0], 12), dtype=np.float64)
    for trip in layers["fc"]["triplets"]:
        o = trip["out_triplet"]
        acc = None
        for g, mat20 in enumerate(trip["color_matrices"]):
            m = np.array(mat20, dtype=np.float64).reshape(4, 5)[:3, :]
            rgb_in = h_anchor[:, 3 * g : 3 * (g + 1)]
            part = q8_np(rgb_in @ m[:, :3].T + m[:, 4].reshape(1, 3))
            acc = part if acc is None else q8_np(acc + part - 0.5)
        u_pad[:, 3 * o : 3 * (o + 1)] = acc

    u_logits = u_pad[:, :10]  # (B, 10) in [0, 1]
    two_m = 2.0 * layers["fc"]["logit_half_range_M"]
    decoded_logits = (u_logits - 0.5) * two_m

    # Stage 6: Exact 8-Bit SVG Softmax (<feMorphology dilate> + <feComponentTransfer>)
    sm = layers["softmax"]
    u_max = np.max(u_logits, axis=1, keepdims=True)  # <feMorphology operator='dilate' radius='10 0'>
    r = sm["diff_scale_r"]
    d = q8_np(r * u_logits - r * u_max + 1.0)
    e = svg_lut_eval(d, sm["exp_table_values"])
    bar_e = q8_np(np.sum(e, axis=1, keepdims=True) / 10.0)
    v_full = svg_lut_eval(bar_e, sm["logsum_table_values"])
    p_log = q8_np(d - sm["logsum_sub_k"] * v_full)
    probs = svg_lut_eval(p_log, sm["exp_table_values"])
    return decoded_logits, probs


export_spec = build_export_dict(model)
print("Compiled SVG filter export spec successfully!")
print(f"  Logit half-range M = +-{export_spec['layers']['fc']['logit_half_range_M']}")

In [ ]:
# ---------------------------------------------------------------------------
# 5. Evaluate Held-Out 10,000-Image MNIST Test Set (Float vs Exact 8-Bit SVG)
# ---------------------------------------------------------------------------

_, test_acc_float = evaluate(test_loader, quantize=False)
_, test_acc_qat = evaluate(test_loader, quantize=True)

all_svg_logits, all_svg_probs = [], []
SIM_BATCH = 500
for start in range(0, len(x_test_np), SIM_BATCH):
    lg, pr = simulate_svg_cnn_batch(x_test_np[start : start + SIM_BATCH], export_spec)
    all_svg_logits.append(lg)
    all_svg_probs.append(pr)

svg_logits = np.concatenate(all_svg_logits, axis=0)
svg_probs = np.concatenate(all_svg_probs, axis=0)
svg_preds = svg_probs.argmax(axis=1)
test_acc_svg = float((svg_preds == y_test_np).mean())

# Compare 8-bit SVG Softmax vs exact float64 Softmax from decoded logits
z_shift = svg_logits - svg_logits.max(axis=1, keepdims=True)
exact_probs = np.exp(z_shift) / np.exp(z_shift).sum(axis=1, keepdims=True)
softmax_max_err = np.max(np.abs(svg_probs - exact_probs), axis=1)

print("=================================================================")
print(f"  MNIST Test Accuracy (Float32):               {test_acc_float * 100:.2f}%")
print(f"  MNIST Test Accuracy (PyTorch 8-bit QAT):     {test_acc_qat * 100:.2f}%")
print(f"  MNIST Test Accuracy (Exact SVG Simulator):   {test_acc_svg * 100:.2f}%")
print(f"  8-Bit SVG Softmax Median / Max Abs Error:    {np.median(softmax_max_err):.4f} / {np.max(softmax_max_err):.4f}")
print("=================================================================")
assert test_acc_svg >= 0.90, f"Expected >= 90% SVG test accuracy, got {test_acc_svg * 100:.2f}%"

# Visualize sample test predictions & SVG Softmax probability distributions
fig, axes = plt.subplots(2, 5, figsize=(13, 5.0))
for idx in range(5):
    img = x_test_np[idx]
    probs = svg_probs[idx]
    pred = int(svg_preds[idx])
    truth = int(y_test_np[idx])

    axes[0, idx].imshow(img, cmap="gray", vmin=0, vmax=1)
    axes[0, idx].set_title(f"True: {truth} | SVG Pred: {pred} ({probs[pred]*100:.0f}%)", fontsize=9.5)
    axes[0, idx].axis("off")

    colors = ["#10b981" if d == pred else "#38bdf8" for d in range(10)]
    axes[1, idx].barh(range(10), probs * 100.0, color=colors)
    axes[1, idx].set_yticks(range(10))
    axes[1, idx].invert_yaxis()
    axes[1, idx].set_xlim(0, 105)
    axes[1, idx].set_xlabel("SVG Softmax %", fontsize=8.5)
    axes[1, idx].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# 6. Export weights.json (with Golden Unseen Test Samples) & Download
# ---------------------------------------------------------------------------

# Bundle 30 unseen held-out test digits (3 per class 0..9) + golden SVG outputs
golden_samples = []
for digit in range(10):
    matches = np.where(y_test_np == digit)[0][:3]
    for rank, test_i in enumerate(matches):
        img_u8 = np.round(np.clip(x_test_np[test_i], 0.0, 1.0) * 255.0).astype(int)
        golden_samples.append({
            "id": f"mnist_test_{digit}_{rank}",
            "test_index": int(test_i),
            "label": int(digit),
            "svg_pred": int(svg_preds[test_i]),
            "svg_probs": [round(float(p), 5) for p in svg_probs[test_i]],
            "svg_logits": [round(float(z), 4) for z in svg_logits[test_i]],
            "pixels_u8": img_u8.tolist(),
        })

export_spec["metrics"] = {
    "val_accuracy_qat": round(float(best_val_acc), 5),
    "test_accuracy_float": round(float(test_acc_float), 5),
    "test_accuracy_qat": round(float(test_acc_qat), 5),
    "test_accuracy_svg_uint8": round(float(test_acc_svg), 5),
    "softmax_median_abs_error": round(float(np.median(softmax_max_err)), 5),
    "total_parameters": int(total_params),
}
export_spec["golden_samples"] = golden_samples

OUT_PATH = "weights.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(export_spec, f, separators=(",", ":"))

size_kb = os.path.getsize(OUT_PATH) / 1024.0
print(f"Saved {OUT_PATH} ({size_kb:.1f} KB) | Exact SVG Test Accuracy: {test_acc_svg * 100:.2f}%")

try:
    from google.colab import files
    files.download(OUT_PATH)
    print("Triggered browser download for weights.json!")
except ImportError:
    print("Not running inside Google Colab; weights.json written to current directory.")